transition testing

In [12]:
import os
import numpy as np
import pandas as pd
from collections import defaultdict, Counter

def parse_screen_id_for_sort(s):
    # screen_id might be "2134" or "2134_something"
    try:
        return int(str(s).split("_")[0])
    except:
        return str(s)

def build_sequences(clusters_tsv):
    df = pd.read_csv(clusters_tsv, sep="\t")
    # group by (app_id, trace_id) and sort by screen_id
    seqs = []
    total_seqs=0
    for (app, trace), g in df.groupby(["app_id", "trace_id"]):
        g2 = g.copy()
        g2["sort_key"] = g2["screen_id"].apply(parse_screen_id_for_sort)
        g2 = g2.sort_values("sort_key")
        seq = g2["cluster_id"].astype(int).tolist()
        # remove consecutive duplicates (optional; helps with identical screens)
        seq_comp = [seq[0]] if seq else []
        for c in seq[1:]:
            if c != seq_comp[-1]:
                seq_comp.append(c)
        total_seqs+=1
        if len(seq_comp) >= 2:
            seqs.append((app, trace, seq_comp))
    print(f"Total sequences: {total_seqs}"
          f", after filtering length>=2: {len(seqs)}")
    return seqs

def split_by_app(seqs, test_ratio=0.2, seed=42):
    apps = sorted(list({a for a, _, _ in seqs}))
    rng = np.random.default_rng(seed)
    rng.shuffle(apps)
    n_test = max(1, int(len(apps) * test_ratio))
    test_apps = set(apps[:n_test])

    train = [s for s in seqs if s[0] not in test_apps]
    test = [s for s in seqs if s[0] in test_apps]
    return train, test, test_apps

def train_bigram_model(train_seqs, num_clusters, smoothing=1.0):
    # Count transitions
    counts = np.zeros((num_clusters, num_clusters), dtype=np.float64)
    for _, _, seq in train_seqs:
        for a, b in zip(seq[:-1], seq[1:]):
            counts[a, b] += 1.0

    # add smoothing
    counts += smoothing
    probs = counts / counts.sum(axis=1, keepdims=True)
    return probs

def recall_at_k(test_seqs, probs, k_list=(1,3,5,10)):
    hits = {k: 0 for k in k_list}
    total = 0

    for _, _, seq in test_seqs:
        for a, b in zip(seq[:-1], seq[1:]):
            total += 1
            row = probs[a]
            top = np.argsort(-row)  # descending
            for k in k_list:
                if b in top[:k]:
                    hits[k] += 1

    return {k: hits[k] / max(1, total) for k in k_list}, total


In [6]:
clusters_tsv="../../processed_data/clusters/screen_clusters.tsv"
test_ratio=0.2
seed=42
smoothing=1.0

In [17]:
seqs = build_sequences(clusters_tsv)
print(f"Traces with length>=2: {len(seqs)}")


Total sequences: 10292, after filtering length>=2: 8096
Traces with length>=2: 8096


In [11]:
seqs[3]

('Kenneth.Currency', 'trace_0', [5, 75, 57, 43, 37, 17])

In [21]:
all_clusters = set()
for _, _, seq in seqs:
    #print(seq)
    all_clusters.update(seq)
num_clusters = max(all_clusters) + 1
print(f"Num clusters (from data): {num_clusters}")
print(all_clusters)



Num clusters (from data): 80
{0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79}


In [ ]:
train, test, test_apps = split_by_app(seqs, test_ratio=test_ratio, seed=seed)
print(f"Train traces: {len(train)} | Test traces: {len(test)} | Test apps: {len(test_apps)}")



Train traces: 6468 | Test traces: 1628 | Test apps: 1503


In [42]:
print(train[69])
print(test[0])
print(list(test_apps)[0])

('app.horoscope1_5.com', 'trace_0', [3, 31, 42, 36, 42, 43, 36, 42, 36, 3, 52, 26, 3, 42, 36, 42, 3])
('air.bahraniapps.comicspanelcreator', 'trace_0', [29, 75, 19])
com.andromo.dev276588.app308783


In [27]:
probs = train_bigram_model(train, num_clusters=num_clusters, smoothing=smoothing)
scores, total_edges = recall_at_k(test, probs, k_list=(1,3,5,10))


In [28]:
print("\n=== Next-step prediction (cluster bigram) ===")
print(f"Test edges: {total_edges}")
for k, v in scores.items():
        print(f"Recall@{k}: {v:.4f}")


=== Next-step prediction (cluster bigram) ===
Test edges: 7560
Recall@1: 0.1142
Recall@3: 0.2316
Recall@5: 0.3213
Recall@10: 0.4746


In [37]:
print(probs.max(), probs.min())
print(probs.shape)

0.23715415019762845 0.0006934812760055479
(80, 80)


In [ ]:

# Save transition matrix (optional)
np.save("transition_probs.npy", probs.astype(np.float32))
print("\nSaved: transition_probs.npy")


In [ ]:


seqs = build_sequences(clusters_tsv)
print(f"Traces with length>=2: {len(seqs)}")

# Determine cluster count
all_clusters = set()
    for _, _, seq in seqs:
        all_clusters.update(seq)
    num_clusters = max(all_clusters) + 1
    print(f"Num clusters (from data): {num_clusters}")

    train, test, test_apps = split_by_app(seqs, test_ratio=test_ratio, seed=seed)
    print(f"Train traces: {len(train)} | Test traces: {len(test)} | Test apps: {len(test_apps)}")

    probs = train_bigram_model(train, num_clusters=num_clusters, smoothing=smoothing)
    scores, total_edges = recall_at_k(test, probs, k_list=(1,3,5,10))

    print("\n=== Next-step prediction (cluster bigram) ===")
    print(f"Test edges: {total_edges}")
    for k, v in scores.items():
        print(f"Recall@{k}: {v:.4f}")

    # Save transition matrix (optional)
    np.save("transition_probs.npy", probs.astype(np.float32))
    print("\nSaved: transition_probs.npy")


In [ ]:
app_category_tsv="../../../../dataset/6- app_details.csv"

cat_df = pd.read_csv(app_category_tsv,encoding="cp1252", low_memory=False)
  
#with open(app_category_tsv, "r", encoding="utf-8", errors="replace") as f:
#        cat_df = pd.read_csv(f, sep="\t", low_memory=False)


UnicodeDecodeError: 'utf-8' codec can't decode byte 0x81 in position 5: invalid start byte